In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_06_accounts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_06_accounts_attributes
# Spec Ref  : §1.9 accounts_attributes (account_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY account.
#             Spec DQ gate: SELECT COUNT(*) FROM accounts_all a
#               LEFT JOIN accounts_attributes aa ON a.id = aa.account_id
#               WHERE aa.account_id IS NULL  ← must return 0
# Run after : map_02 (accounts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)
print(f"=== Map 06: accounts_attributes ===  customer={customer_id}  tenant={tenant_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"
initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.accounts_attributes",
    """
        account_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="account_id"
)

In [0]:
# CELL 4 ── CDF-Aware Incremental MERGE for accounts_attributes (tenant-scoped)
# Instead of scanning ALL accounts_all, reads CDF to find only new/deleted accounts.
# New accounts get default attributes; removed accounts are deleted.
try:
    safe_drop_view(f"{sv}.accounts_attributes")

    # ── VERSION TRACKING: last-processed source version ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.accounts_attributes ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' and 'does not have property' not in str(val) else 0
    except Exception:
        last_source_ver = 0

    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.accounts_all)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.accounts_all  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.accounts_attributes WHERE tenant = {tenant_id}").collect()[0][0]

    if target_count == 0:
        # First run: full MERGE (same as original)
        print(f"  First run — full MERGE from accounts_all")
        spark.sql(f"""
            MERGE INTO {sv}.accounts_attributes AS target
            USING (
                SELECT id AS account_id, tenant
                FROM {sv}.accounts_all
                WHERE tenant = {tenant_id}
            ) AS source
            ON target.account_id = source.account_id AND target.tenant = source.tenant
            WHEN NOT MATCHED THEN INSERT (account_id, tenant, is_paying, is_excluded, mrr)
                VALUES (source.account_id, source.tenant, false, false, 0.0)
            WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
        """)
    elif last_source_ver >= source_max_ver:
        print(f"  Source unchanged (ver {source_max_ver}) — skipping")
    else:
        # Incremental: read CDF from accounts_all
        start_ver = last_source_ver + 1
        try:
            cdf_df = (spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.accounts_all")
                .filter(f"tenant = {tenant_id}")
            )

            # Count by change type
            from pyspark.sql import functions as F
            change_counts = {row['_change_type']: row['count'] for row in cdf_df.groupBy('_change_type').count().collect()}
            print(f"  CDF change types: {change_counts}")

            # INSERT attributes for new accounts
            new_accounts = cdf_df.filter("_change_type = 'insert'").select(F.col("id").alias("account_id"), "tenant")
            new_count = new_accounts.count()
            if new_count > 0:
                new_accounts.createOrReplaceTempView("new_accts_cdf")
                spark.sql(f"""
                    MERGE INTO {sv}.accounts_attributes AS target
                    USING new_accts_cdf AS source
                    ON target.account_id = source.account_id AND target.tenant = source.tenant
                    WHEN NOT MATCHED THEN INSERT (account_id, tenant, is_paying, is_excluded, mrr)
                        VALUES (source.account_id, source.tenant, false, false, 0.0)
                """)
                print(f"  Inserted {new_count} new account attributes")

            # DELETE attributes for removed accounts
            deleted_accounts = cdf_df.filter("_change_type = 'delete'").select(F.col("id").alias("account_id"), "tenant")
            del_count = deleted_accounts.count()
            if del_count > 0:
                deleted_accounts.createOrReplaceTempView("deleted_accts_cdf")
                spark.sql(f"""
                    MERGE INTO {sv}.accounts_attributes AS target
                    USING deleted_accts_cdf AS source
                    ON target.account_id = source.account_id AND target.tenant = source.tenant
                    WHEN MATCHED THEN DELETE
                """)
                print(f"  Deleted {del_count} removed account attributes")

            if new_count == 0 and del_count == 0:
                print(f"  CDF: only updates detected — no attribute changes needed")

        except Exception as cdf_err:
            print(f"  CDF read failed ({cdf_err}), falling back to full MERGE")
            spark.sql(f"""
                MERGE INTO {sv}.accounts_attributes AS target
                USING (
                    SELECT id AS account_id, tenant
                    FROM {sv}.accounts_all
                    WHERE tenant = {tenant_id}
                ) AS source
                ON target.account_id = source.account_id AND target.tenant = source.tenant
                WHEN NOT MATCHED THEN INSERT (account_id, tenant, is_paying, is_excluded, mrr)
                    VALUES (source.account_id, source.tenant, false, false, 0.0)
                WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
            """)

    # Save version
    spark.sql(f"ALTER TABLE {sv}.accounts_attributes SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

    n = cnt(f"{sv}.accounts_attributes")
    print(f"  accounts_attributes: {n:,} rows")
except Exception as e:
    print(f"\u274c accounts_attributes build failed: {e}")
    log_audit(
        job_name       = "02b_map_06_accounts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 5 -- Spec DQ gate: every account must have an attributes row
try:
    uncovered = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all a
        LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
        WHERE aa.account_id IS NULL
    """).collect()[0][0]

    n = cnt(f"{sv}.accounts_attributes")
    print(f"  accounts_attributes: {n:,} rows")
    status = '✅ PASS' if uncovered == 0 else '❌ FAIL'
    print(f"  Uncovered accounts: {uncovered}  (spec gate = 0)  {status}")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_06_accounts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise